# Dendritron Tissue Compiler v0.5

Single-cell runnable benchmark. Execute the code cell in Google Colab or Jupyter.

In [ ]:
"""
Dendritron Tissue Compiler v0.5

Automatic discovery of reusable local abstractions.

Given only complete Boolean task tables, the compiler searches bounded local
input subsets for exact XOR functional decompositions:

    F_t(x) = A(x_S) XOR L_t(x_not_S)

It groups tasks that share the same canonical A, compiles A once as an intact
Dendritron region, compiles each L_t locally, and connects them through one
shared XOR Dendritron. Tasks unsupported by an exact decomposition remain
monolithic rather than being forced into a false shared space.

The benchmark then changes four tasks to a common specialized A' and checks
whether the compiler creates a new reusable version while preserving A.

This is a finite Boolean architecture experiment, not a general factorization
algorithm or a real-world superiority claim.
"""

from __future__ import annotations

from dataclasses import dataclass
from itertools import combinations
from pathlib import Path
from typing import Dict, Iterable, List, Mapping, Sequence, Tuple
import hashlib
import json

import numpy as np
import pandas as pd

SEED = 41
try:
    OUTPUT_DIR = Path(__file__).resolve().parent
except NameError:
    OUTPUT_DIR = Path.cwd()


def all_binary_inputs(bits: int) -> np.ndarray:
    values = np.arange(2**bits, dtype=np.uint64)
    shifts = np.arange(bits - 1, -1, -1, dtype=np.uint64)
    return ((values[:, None] >> shifts[None, :]) & 1).astype(np.uint8)


def bits_to_index(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.uint8)
    powers = (2 ** np.arange(x.shape[1] - 1, -1, -1)).astype(np.int64)
    return x.astype(np.int64) @ powers


def balanced_truth_table(bits: int, rng: np.random.Generator) -> np.ndarray:
    n = 2**bits
    table = np.zeros(n, dtype=np.uint8)
    table[rng.choice(n, size=n // 2, replace=False)] = 1
    return table


def specialize_balanced(
    base: np.ndarray,
    rng: np.random.Generator,
    swaps: int,
) -> np.ndarray:
    base = np.asarray(base, dtype=np.uint8)
    result = base.copy()
    ones = np.flatnonzero(result == 1)
    zeros = np.flatnonzero(result == 0)
    result[rng.choice(ones, size=swaps, replace=False)] = 0
    result[rng.choice(zeros, size=swaps, replace=False)] = 1
    return result


def table_hash(table: np.ndarray) -> str:
    return hashlib.sha256(np.asarray(table, dtype=np.uint8).tobytes()).hexdigest()


def branch_count(table: np.ndarray) -> int:
    return int(np.sum(np.asarray(table, dtype=np.uint8)))


def literal_cost(table: np.ndarray, input_bits: int) -> int:
    return branch_count(table) * input_bits


def compose_xor_table(
    all_x: np.ndarray,
    subset: Sequence[int],
    shared_table: np.ndarray,
    local_table: np.ndarray,
) -> np.ndarray:
    subset = tuple(subset)
    complement = tuple(i for i in range(all_x.shape[1]) if i not in subset)
    shared_values = np.asarray(shared_table, dtype=np.uint8)[
        bits_to_index(all_x[:, subset])
    ]
    local_values = np.asarray(local_table, dtype=np.uint8)[
        bits_to_index(all_x[:, complement])
    ]
    return shared_values ^ local_values


def xor_decompose(
    table: np.ndarray,
    all_x: np.ndarray,
    subset: Sequence[int],
) -> Tuple[np.ndarray, np.ndarray] | None:
    """Return canonical A, L when F=A XOR L exactly.

    Canonicalization sets A(0...0)=0. This removes the A/L simultaneous
    complement ambiguity and makes shared factors content-addressable.
    """
    subset = tuple(subset)
    complement = tuple(i for i in range(all_x.shape[1]) if i not in subset)
    row_index = bits_to_index(all_x[:, subset])
    col_index = bits_to_index(all_x[:, complement])
    matrix = np.empty((2 ** len(subset), 2 ** len(complement)), dtype=np.uint8)
    matrix[row_index, col_index] = np.asarray(table, dtype=np.uint8)

    local = matrix[0, :].copy()
    shared = matrix[:, 0] ^ matrix[0, 0]
    reconstruction = shared[:, None] ^ local[None, :]
    if np.array_equal(matrix, reconstruction):
        return shared.astype(np.uint8), local.astype(np.uint8)
    return None


@dataclass
class CandidateFamily:
    subset: Tuple[int, ...]
    task_ids: Tuple[int, ...]
    shared_table: np.ndarray
    local_tables: Dict[int, np.ndarray]
    monolithic_cost: int
    factored_cost: int
    savings: int


@dataclass
class CompiledFamily:
    family_id: str
    subset: Tuple[int, ...]
    task_ids: Tuple[int, ...]
    shared_table: np.ndarray
    local_tables: Dict[int, np.ndarray]
    savings: int

    @property
    def shared_signature(self) -> str:
        return table_hash(self.shared_table)

    @property
    def complement(self) -> Tuple[int, ...]:
        n_bits = len(self.subset) + int(round(np.log2(len(next(iter(self.local_tables.values()))))))
        return tuple(i for i in range(n_bits) if i not in self.subset)


class AutomaticTissueCompiler:
    def __init__(
        self,
        n_bits: int,
        *,
        min_local_arity: int = 1,
        max_local_arity: int = 4,
        minimum_family_size: int = 2,
    ) -> None:
        self.n_bits = int(n_bits)
        self.min_local_arity = int(min_local_arity)
        self.max_local_arity = int(max_local_arity)
        self.minimum_family_size = int(minimum_family_size)
        self.all_x = all_binary_inputs(n_bits)
        self.families: List[CompiledFamily] = []
        self.monolithic_tasks: Dict[int, np.ndarray] = {}
        self.task_to_family: Dict[int, str] = {}
        self.candidate_rows: List[dict] = []

    @property
    def candidate_subset_count(self) -> int:
        return sum(
            math_comb(self.n_bits, k)
            for k in range(self.min_local_arity, self.max_local_arity + 1)
        )

    def _candidate_families(
        self,
        task_tables: Mapping[int, np.ndarray],
        remaining: Iterable[int],
    ) -> List[CandidateFamily]:
        remaining = sorted(remaining)
        candidates: List[CandidateFamily] = []
        for arity in range(self.min_local_arity, self.max_local_arity + 1):
            for subset in combinations(range(self.n_bits), arity):
                grouped: Dict[bytes, List[int]] = {}
                decompositions: Dict[int, Tuple[np.ndarray, np.ndarray]] = {}
                for task_id in remaining:
                    result = xor_decompose(task_tables[task_id], self.all_x, subset)
                    if result is None:
                        continue
                    shared, local = result
                    # Constant factors are vacuous and should not create a region.
                    if np.all(shared == shared[0]):
                        continue
                    decompositions[task_id] = (shared, local)
                    grouped.setdefault(shared.tobytes(), []).append(task_id)

                for shared_bytes, task_ids in grouped.items():
                    if len(task_ids) < self.minimum_family_size:
                        continue
                    task_ids = sorted(task_ids)
                    shared = decompositions[task_ids[0]][0]
                    local_tables = {
                        task_id: decompositions[task_id][1] for task_id in task_ids
                    }
                    mono = sum(
                        literal_cost(task_tables[task_id], self.n_bits)
                        for task_id in task_ids
                    )
                    # One shared factor + one local factor per task + one XOR node.
                    factored = literal_cost(shared, arity)
                    factored += sum(
                        literal_cost(local_tables[task_id], self.n_bits - arity)
                        for task_id in task_ids
                    )
                    factored += 4  # two two-literal XOR branches
                    candidate = CandidateFamily(
                        subset=tuple(subset),
                        task_ids=tuple(task_ids),
                        shared_table=shared,
                        local_tables=local_tables,
                        monolithic_cost=mono,
                        factored_cost=factored,
                        savings=mono - factored,
                    )
                    candidates.append(candidate)
                    self.candidate_rows.append(
                        {
                            "subset": str(tuple(subset)),
                            "arity": arity,
                            "task_count": len(task_ids),
                            "task_ids": ",".join(map(str, task_ids)),
                            "shared_signature": table_hash(shared),
                            "monolithic_literal_cost": mono,
                            "factored_literal_cost": factored,
                            "savings": mono - factored,
                        }
                    )
        return candidates

    def compile(self, task_tables: Mapping[int, np.ndarray]) -> "AutomaticTissueCompiler":
        self.families = []
        self.monolithic_tasks = {}
        self.task_to_family = {}
        self.candidate_rows = []
        remaining = set(task_tables)
        family_number = 0

        while True:
            candidates = self._candidate_families(task_tables, remaining)
            candidates = [candidate for candidate in candidates if candidate.savings > 0]
            if not candidates:
                break
            candidates.sort(
                key=lambda c: (
                    c.savings,
                    len(c.task_ids),
                    -len(c.subset),
                    tuple(-i for i in c.subset),
                ),
                reverse=True,
            )
            best = candidates[0]
            family_number += 1
            family_id = f"family_{family_number}"
            family = CompiledFamily(
                family_id=family_id,
                subset=best.subset,
                task_ids=best.task_ids,
                shared_table=best.shared_table.copy(),
                local_tables={k: v.copy() for k, v in best.local_tables.items()},
                savings=best.savings,
            )
            self.families.append(family)
            for task_id in best.task_ids:
                self.task_to_family[task_id] = family_id
            remaining -= set(best.task_ids)

        self.monolithic_tasks = {
            task_id: np.asarray(task_tables[task_id], dtype=np.uint8).copy()
            for task_id in sorted(remaining)
        }
        return self

    def family_for_task(self, task_id: int) -> CompiledFamily | None:
        family_id = self.task_to_family.get(task_id)
        if family_id is None:
            return None
        return next(f for f in self.families if f.family_id == family_id)

    def reconstructed_table(self, task_id: int) -> np.ndarray:
        family = self.family_for_task(task_id)
        if family is None:
            return self.monolithic_tasks[task_id].copy()
        return compose_xor_table(
            self.all_x,
            family.subset,
            family.shared_table,
            family.local_tables[task_id],
        )

    def evaluate(self, task_tables: Mapping[int, np.ndarray]) -> pd.DataFrame:
        rows = []
        for task_id in sorted(task_tables):
            reconstructed = self.reconstructed_table(task_id)
            target = np.asarray(task_tables[task_id], dtype=np.uint8)
            family = self.family_for_task(task_id)
            rows.append(
                {
                    "task_id": task_id,
                    "accuracy": float(np.mean(reconstructed == target)),
                    "compiled_as": "family" if family is not None else "monolithic",
                    "family_id": family.family_id if family is not None else "",
                    "subset": str(family.subset) if family is not None else "",
                    "shared_signature": family.shared_signature if family is not None else "",
                }
            )
        return pd.DataFrame(rows)

    def storage(self, task_tables: Mapping[int, np.ndarray]) -> dict:
        mono_branches = sum(branch_count(table) for table in task_tables.values())
        mono_literals = sum(
            literal_cost(table, self.n_bits) for table in task_tables.values()
        )

        # Per-task factorization keeps the local decomposition but duplicates
        # the shared factor for every consumer.
        independent_branches = 2  # one shared XOR Dendritron
        independent_literals = 4
        for family in self.families:
            arity = len(family.subset)
            for task_id in family.task_ids:
                independent_branches += branch_count(family.shared_table)
                independent_branches += branch_count(family.local_tables[task_id])
                independent_literals += literal_cost(family.shared_table, arity)
                independent_literals += literal_cost(
                    family.local_tables[task_id], self.n_bits - arity
                )
        for table in self.monolithic_tasks.values():
            independent_branches += branch_count(table)
            independent_literals += literal_cost(table, self.n_bits)

        tissue_branches = 2  # one immutable XOR region reused by all families
        tissue_literals = 4
        for family in self.families:
            arity = len(family.subset)
            tissue_branches += branch_count(family.shared_table)
            tissue_literals += literal_cost(family.shared_table, arity)
            for local in family.local_tables.values():
                tissue_branches += branch_count(local)
                tissue_literals += literal_cost(local, self.n_bits - arity)
        for table in self.monolithic_tasks.values():
            tissue_branches += branch_count(table)
            tissue_literals += literal_cost(table, self.n_bits)

        return {
            "monolithic_branch_count": mono_branches,
            "monolithic_literal_cost": mono_literals,
            "independent_factored_branch_count": independent_branches,
            "independent_factored_literal_cost": independent_literals,
            "shared_tissue_branch_count": tissue_branches,
            "shared_tissue_literal_cost": tissue_literals,
            "literal_compression_vs_monolithic": mono_literals / tissue_literals,
            "literal_savings_vs_independent_factorization": independent_literals
            - tissue_literals,
        }

    def family_rows(self) -> List[dict]:
        rows = []
        for family in self.families:
            rows.append(
                {
                    "family_id": family.family_id,
                    "subset": str(family.subset),
                    "arity": len(family.subset),
                    "task_count": len(family.task_ids),
                    "task_ids": ",".join(map(str, family.task_ids)),
                    "shared_signature": family.shared_signature,
                    "shared_branch_count": branch_count(family.shared_table),
                    "shared_literal_cost": literal_cost(
                        family.shared_table, len(family.subset)
                    ),
                    "compiler_savings": family.savings,
                }
            )
        return rows


def math_comb(n: int, k: int) -> int:
    from math import comb

    return comb(n, k)


def make_benchmark_tasks() -> Tuple[
    Dict[int, np.ndarray],
    Dict[int, str],
    Dict[str, Tuple[int, ...]],
    Dict[str, np.ndarray],
    Dict[int, np.ndarray],
]:
    rng = np.random.default_rng(SEED)
    n_bits = 8
    all_x = all_binary_inputs(n_bits)
    subset_a = (0, 2, 5, 7)
    subset_b = (1, 3, 4, 6)
    shared_a = balanced_truth_table(4, rng)
    shared_b = balanced_truth_table(4, rng)

    task_tables: Dict[int, np.ndarray] = {}
    labels: Dict[int, str] = {}
    local_tables: Dict[int, np.ndarray] = {}

    for task_id in range(16):
        local = balanced_truth_table(4, rng)
        task_tables[task_id] = compose_xor_table(
            all_x, subset_a, shared_a, local
        )
        labels[task_id] = "A"
        local_tables[task_id] = local

    for task_id in range(16, 24):
        local = balanced_truth_table(4, rng)
        task_tables[task_id] = compose_xor_table(
            all_x, subset_b, shared_b, local
        )
        labels[task_id] = "B"
        local_tables[task_id] = local

    # Deliberately unstructured tasks. The compiler must leave them monolithic.
    for task_id in range(24, 28):
        task_tables[task_id] = balanced_truth_table(8, rng)
        labels[task_id] = f"outlier_{task_id}"

    return (
        task_tables,
        labels,
        {"A": subset_a, "B": subset_b},
        {"A": shared_a, "B": shared_b},
        local_tables,
    )


def group_set(compiler: AutomaticTissueCompiler) -> set[frozenset[int]]:
    return {frozenset(family.task_ids) for family in compiler.families}


def run() -> dict:
    task_tables, labels, true_subsets, true_shared, local_tables = make_benchmark_tasks()
    compiler = AutomaticTissueCompiler(8, min_local_arity=1, max_local_arity=4)
    compiler.compile(task_tables)
    evaluation = compiler.evaluate(task_tables)
    storage = compiler.storage(task_tables)

    true_groups = {
        frozenset(task_id for task_id, label in labels.items() if label == "A"),
        frozenset(task_id for task_id, label in labels.items() if label == "B"),
    }
    recovered_groups = group_set(compiler)
    groups_recovered_exactly = recovered_groups == true_groups

    subset_by_group = {
        frozenset(family.task_ids): family.subset for family in compiler.families
    }
    subsets_recovered_exactly = (
        subset_by_group[
            frozenset(task_id for task_id, label in labels.items() if label == "A")
        ]
        == true_subsets["A"]
        and subset_by_group[
            frozenset(task_id for task_id, label in labels.items() if label == "B")
        ]
        == true_subsets["B"]
    )
    outliers = {task_id for task_id, label in labels.items() if label.startswith("outlier")}
    outliers_left_monolithic = set(compiler.monolithic_tasks) == outliers

    assert float(evaluation.accuracy.min()) == 1.0
    assert groups_recovered_exactly
    assert subsets_recovered_exactly
    assert outliers_left_monolithic

    # ------------------------------------------------------------------
    # Incremental specialization: four members of A now require the same A'.
    rng = np.random.default_rng(SEED + 1)
    specialized_shared = specialize_balanced(true_shared["A"], rng, swaps=2)
    specialized_tasks = {0, 1, 2, 3}
    updated_tables = {task_id: table.copy() for task_id, table in task_tables.items()}
    all_x = all_binary_inputs(8)
    for task_id in specialized_tasks:
        updated_tables[task_id] = compose_xor_table(
            all_x,
            true_subsets["A"],
            specialized_shared,
            local_tables[task_id],
        )

    specialized_compiler = AutomaticTissueCompiler(
        8, min_local_arity=1, max_local_arity=4
    )
    specialized_compiler.compile(updated_tables)
    specialized_eval = specialized_compiler.evaluate(updated_tables)
    specialized_storage = specialized_compiler.storage(updated_tables)

    expected_specialized_groups = {
        frozenset(specialized_tasks),
        frozenset(range(4, 16)),
        frozenset(range(16, 24)),
    }
    specialization_groups_exact = group_set(specialized_compiler) == expected_specialized_groups

    original_a_signature = table_hash(true_shared["A"])
    specialized_signature = table_hash(specialized_shared)
    recovered_signatures = {
        family.shared_signature for family in specialized_compiler.families
    }
    original_a_preserved = original_a_signature in recovered_signatures
    specialized_version_created = specialized_signature in recovered_signatures
    old_a_consumers_accuracy = float(
        specialized_eval.loc[
            specialized_eval.task_id.isin(list(range(4, 16))), "accuracy"
        ].mean()
    )

    assert float(specialized_eval.accuracy.min()) == 1.0
    assert specialization_groups_exact
    assert original_a_preserved
    assert specialized_version_created
    assert old_a_consumers_accuracy == 1.0

    # Results tables.
    families = pd.DataFrame(compiler.family_rows())
    specialized_families = pd.DataFrame(specialized_compiler.family_rows())
    candidate_frame = pd.DataFrame(compiler.candidate_rows).sort_values(
        ["savings", "task_count"], ascending=False
    )
    top_candidates = candidate_frame.head(50).copy()

    storage_frame = pd.DataFrame(
        [
            {"stage": "initial", **storage},
            {"stage": "after_specialization", **specialized_storage},
        ]
    )

    assignments = evaluation.copy()
    assignments["ground_truth_family"] = assignments.task_id.map(labels)
    specialized_assignments = specialized_eval.copy()
    specialized_assignments["expected_specialized"] = specialized_assignments.task_id.isin(
        specialized_tasks
    )

    families.to_csv(OUTPUT_DIR / "dendritron_v0_5_discovered_families.csv", index=False)
    assignments.to_csv(OUTPUT_DIR / "dendritron_v0_5_task_assignments.csv", index=False)
    top_candidates.to_csv(OUTPUT_DIR / "dendritron_v0_5_top_candidates.csv", index=False)
    storage_frame.to_csv(OUTPUT_DIR / "dendritron_v0_5_storage.csv", index=False)
    specialized_families.to_csv(
        OUTPUT_DIR / "dendritron_v0_5_specialized_families.csv", index=False
    )
    specialized_assignments.to_csv(
        OUTPUT_DIR / "dendritron_v0_5_specialized_assignments.csv", index=False
    )

    summary = {
        "seed": SEED,
        "task_count": len(task_tables),
        "input_bits": 8,
        "candidate_subsets_searched": compiler.candidate_subset_count,
        "initial": {
            "families_discovered": len(compiler.families),
            "family_sizes": [len(f.task_ids) for f in compiler.families],
            "groups_recovered_exactly": groups_recovered_exactly,
            "subsets_recovered_exactly": subsets_recovered_exactly,
            "outliers_left_monolithic": outliers_left_monolithic,
            "minimum_accuracy": float(evaluation.accuracy.min()),
            "mean_accuracy": float(evaluation.accuracy.mean()),
            **storage,
        },
        "specialization": {
            "specialized_tasks": sorted(specialized_tasks),
            "shared_states_changed": int(
                np.sum(specialized_shared != true_shared["A"])
            ),
            "families_discovered": len(specialized_compiler.families),
            "family_sizes": [len(f.task_ids) for f in specialized_compiler.families],
            "specialization_groups_exact": specialization_groups_exact,
            "original_shared_version_preserved": original_a_preserved,
            "specialized_version_created": specialized_version_created,
            "old_consumer_accuracy": old_a_consumers_accuracy,
            "minimum_accuracy": float(specialized_eval.accuracy.min()),
            **specialized_storage,
        },
    }
    with open(OUTPUT_DIR / "dendritron_v0_5_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print("DENDRITRON TISSUE COMPILER v0.5")
    print("=" * 76)
    print(f"Candidate local subsets searched: {compiler.candidate_subset_count}")
    print("\nDiscovered families")
    print(families.to_string(index=False))
    print("\nInitial verification")
    print(
        f"Exact groups: {groups_recovered_exactly}; exact subsets: "
        f"{subsets_recovered_exactly}; outliers monolithic: "
        f"{outliers_left_monolithic}; min accuracy: {evaluation.accuracy.min():.2%}"
    )
    print("\nStorage")
    print(storage_frame.to_string(index=False))
    print("\nAfter four tasks specialize into the same new shared version")
    print(specialized_families.to_string(index=False))
    print(
        f"Original A preserved: {original_a_preserved}; specialized A' created: "
        f"{specialized_version_created}; old consumers: "
        f"{old_a_consumers_accuracy:.2%}; min accuracy: "
        f"{specialized_eval.accuracy.min():.2%}"
    )
    print("\nSummary written to dendritron_v0_5_summary.json")
    return summary


if __name__ == "__main__":
    run()
